# DPCNN

Seminar 2. DPCNN (Johnson & Zhang, 2017) trained with the same setup as the RNN models in `03_rnn_models.ipynb`: frozen fastText 300d, `comment_heavy`, max length 200, 5 stratified folds (seed 42), 4 epochs, threshold 0.5.

Runtime -> Change runtime type -> GPU (T4).

## 1. Setup

In [ ]:
BRANCH = "main"
!git clone -q -b {BRANCH} https://github.com/mnhhuyen/AI4SE_toxic_comments.git
%cd AI4SE_toxic_comments
!pip install -q -e ".[dev]" tabulate
!python -m pytest -q tests/test_dpcnn.py 2>&1 | tail -3

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU!")

## 2. Download fastText vectors

In [ ]:
!mkdir -p data/embeddings/fasttext
!wget -q -c https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip -P data/embeddings/fasttext/
!unzip -q -n data/embeddings/fasttext/wiki-news-300d-1M.vec.zip -d data/embeddings/fasttext/
!ls -lh data/embeddings/fasttext/

## 3. Clean data and write the 5 folds

In [ ]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd
from toxic_comments.cleaning import process_cleaning, validate_training_data
from toxic_comments.config import (ID_COLUMN, K_FOLD_DATA_DIR, LABEL_COLUMNS, MODELS_DIR,
                                   PROCESSED_DATA_DIR, RAW_DATA_DIR, RESULTS_DIR)
from toxic_comments.splits import save_kfold_datasets

clean_path = PROCESSED_DATA_DIR / "train_clean.csv"
if clean_path.exists():
    data = pd.read_csv(clean_path)
else:
    data = process_cleaning(validate_training_data(pd.read_csv(RAW_DATA_DIR / "train.csv")))
    clean_path.parent.mkdir(parents=True, exist_ok=True)
    data.to_csv(clean_path, index=False)

for column in ["comment_text", "comment_light", "comment_heavy"]:
    data[column] = data[column].fillna("").astype(str)
data = data[data["is_empty_heavy"] == 0].reset_index(drop=True)

keep = [ID_COLUMN, "comment_text", "comment_light", "comment_heavy", *LABEL_COLUMNS]
save_kfold_datasets(data[keep], n_splits=5, strategy="stratified")
print(data.shape)

## 4. Train DPCNN on 5 folds

In [ ]:
from toxic_comments.train import train_from_folds

NAME = "dpcnn"
if (RESULTS_DIR / f"{NAME}_fold_metrics.csv").exists():
    print(f"skip {NAME}: results already exist")
else:
    started = time.perf_counter()
    train_from_folds(folds_dir=K_FOLD_DATA_DIR, model_name=NAME, output_dir=MODELS_DIR)
    dpcnn_minutes = (time.perf_counter() - started) / 60
    print(f"{NAME}: {dpcnn_minutes:.1f} min for 5 folds")

## 5. Results vs the RNN models (mean ± std over 5 folds)

In [ ]:
from toxic_comments.evaluation import summarize_folds

MODELS = ["dpcnn", "birnn_max_attention", "bilstm_max_attention", "bigru_shared_max_attention"]
NICE = {"dpcnn": "DPCNN", "birnn_max_attention": "Bi-RNN",
        "bilstm_max_attention": "Bi-LSTM", "bigru_shared_max_attention": "Bi-GRU"}

folds = pd.concat([pd.read_csv(RESULTS_DIR / f"{n}_fold_metrics.csv") for n in MODELS])
key = ["micro_f1", "macro_f1", "micro_roc_auc", "macro_roc_auc"]
summary = summarize_folds(folds)[key].rename(index=NICE).sort_values(("macro_f1", "mean"), ascending=False)
print(summary.round(4).to_markdown())

In [ ]:
per_label = pd.concat([pd.read_csv(RESULTS_DIR / f"{n}_fold_per_label.csv") for n in MODELS])
per_label["model_name"] = per_label["model_name"].map(NICE)
order = ["DPCNN", "Bi-RNN", "Bi-LSTM", "Bi-GRU"]
f1_table = per_label.groupby(["label", "model_name"])["f1"].mean().unstack()[order].round(3)
auc_table = per_label.groupby(["label", "model_name"])["roc_auc"].mean().unstack()[order].round(3)
print("### F1 per label\n", f1_table.to_markdown())
print("\n### ROC-AUC per label\n", auc_table.to_markdown())

In [ ]:
import matplotlib.pyplot as plt

folds["model"] = folds["model_name"].map(NICE)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ["macro_roc_auc", "macro_f1"]):
    stats = folds.groupby("model")[metric].agg(["mean", "std"]).loc[order]
    colors = ["#d1495b" if m == "DPCNN" else "#8d99ae" for m in stats.index]
    ax.bar(stats.index, stats["mean"], yerr=stats["std"], capsize=4, color=colors)
    ax.set_title(f"{metric} (mean ± std, 5 folds)")
    ax.set_ylim(stats["mean"].min() - 3 * stats["std"].max() - 0.01, min(1, stats["mean"].max() + 0.01))
fig.tight_layout()
fig.savefig(RESULTS_DIR / "dpcnn_vs_rnn.png", dpi=150)
plt.show()

## 6. Training loss, fold 1

In [ ]:
import joblib

for name in MODELS:
    path = MODELS_DIR / "fold_1" / f"{name}.joblib"
    if not path.exists():
        print("missing", path); continue
    model = joblib.load(path)
    plt.plot(range(1, len(model.history_) + 1), model.history_, marker="o", label=NICE[name])
plt.xlabel("epoch"); plt.ylabel("train BCE loss"); plt.legend(); plt.title("Training loss, fold 1")
plt.savefig(RESULTS_DIR / "dpcnn_loss_curves.png", dpi=150)
plt.show()

dpcnn = joblib.load(MODELS_DIR / "fold_1" / "dpcnn.joblib")
print(f"fastText coverage of vocab = {dpcnn.embedding_coverage_:.1%}")
print("trainable parameters:", sum(p.numel() for p in dpcnn.model_.parameters() if p.requires_grad))

## 7. Inference time on fold 1 test set

In [ ]:
test = pd.read_csv(K_FOLD_DATA_DIR / "fold_1" / "test.csv")
test["comment_heavy"] = test["comment_heavy"].fillna("").astype(str)
for name in MODELS:
    path = MODELS_DIR / "fold_1" / f"{name}.joblib"
    if not path.exists():
        print("missing", path); continue
    model = joblib.load(path)
    started = time.perf_counter(); model.predict_proba(test["comment_heavy"])
    print(f"{NICE[name]:8s} {time.perf_counter() - started:6.1f}s for {len(test):,} comments")

## 8. Error analysis on fold 1 test set

In [ ]:
scores = dpcnn.predict_proba(test["comment_heavy"])
test["toxic_prob"] = scores[:, 0]
pd.set_option("display.max_colwidth", 120)
print("False negatives (toxic but low score):")
display(test[test["toxic"] == 1].nsmallest(5, "toxic_prob")[["comment_text", "toxic_prob"]])
print("False positives (clean but high score):")
display(test[test["toxic"] == 0].nlargest(5, "toxic_prob")[["comment_text", "toxic_prob"]])

## 9. Download results

In [ ]:
!zip -q -r dpcnn_results.zip results/dpcnn_* results/rnn_*
from google.colab import files
files.download("dpcnn_results.zip")